# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhavanathiru13-droid/flyrank-AI_1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule:

Prioritize pages that had meaningful search visibility in February but showed a weak click-through rate and/or a weak average search position. Higher visibility receives more priority because improving a visible page can have more practical value.

Reason codes:
- visible_low_ctr: meaningful impressions with low CTR
- visible_poor_position: meaningful impressions with average position worse than page 1
- visible_low_ctr_poor_position: both signals are present
- insufficient_signal: not enough search data to make a strong review recommendation

Signal checks:

1. CTR vs position — CONFIRMED
Low-CTR pages are common beyond page 1, while higher-CTR pages are more concentrated on page 1 and in the top 3.

2. Search visibility volume — CONFIRMED
The February data contains a clear set of high-impression pages, so impression volume is useful for prioritizing pages with meaningful search visibility.

In [6]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{get_hf_token()}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("Connected to FlyRank warehouse")
print("Using February 2026 as the feature window.")

# Signal 1: CTR vs search position
ctr_position = con.execute(f"""
WITH base AS (
    SELECT
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {FEB}
    WHERE gsc_data_available = TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_impressions > 0
      AND gsc_avg_position IS NOT NULL
)
SELECT
    CASE
        WHEN gsc_clicks * 1.0 / gsc_impressions < 0.02 THEN 'low_ctr'
        WHEN gsc_clicks * 1.0 / gsc_impressions < 0.05 THEN 'mid_ctr'
        ELSE 'high_ctr'
    END AS ctr_bucket,
    CASE
        WHEN gsc_avg_position <= 3 THEN 'top_3'
        WHEN gsc_avg_position <= 10 THEN 'page_1'
        ELSE 'beyond_page_1'
    END AS position_bucket,
    COUNT(*) AS n
FROM base
GROUP BY 1, 2
ORDER BY 1, 2
""").fetchdf()

print("\nSignal 1 — CTR vs position")
display(ctr_position)

# Signal 2: search visibility / impression volume
volume = con.execute(f"""
SELECT
    CASE
        WHEN gsc_impressions < 100 THEN 'low_volume'
        WHEN gsc_impressions < 500 THEN 'mid_volume'
        ELSE 'high_volume'
    END AS volume_bucket,
    COUNT(*) AS n
FROM {FEB}
WHERE gsc_data_available = TRUE
  AND gsc_impressions IS NOT NULL
  AND gsc_impressions > 0
GROUP BY 1
ORDER BY 1
""").fetchdf()

print("\nSignal 2 — Search visibility volume")
display(volume)

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse
Using February 2026 as the feature window.

Signal 1 — CTR vs position


,ctr_bucket,position_bucket,n
0,high_ctr,beyond_page_1,6328
1,high_ctr,page_1,12380
2,high_ctr,top_3,7074
3,low_ctr,beyond_page_1,848477
4,low_ctr,page_1,1082538
5,low_ctr,top_3,616690
6,mid_ctr,beyond_page_1,9635
7,mid_ctr,page_1,26437
8,mid_ctr,top_3,12223



Signal 2 — Search visibility volume


,volume_bucket,n
0,high_volume,59487
1,low_volume,2192382
2,mid_volume,369914


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Build the transparent baseline score and write the ranked queue.

import os

os.makedirs("/content/work/outputs", exist_ok=True)

queue_sql = f"""
WITH latest_feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ROW_NUMBER() OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date DESC
        ) AS rn
    FROM {FEB}
),

base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_data_available,
        COALESCE(gsc_impressions, 0) AS impressions,
        COALESCE(gsc_clicks, 0) AS clicks,
        gsc_avg_position,

        CASE
            WHEN gsc_data_available = TRUE
                 AND gsc_impressions IS NOT NULL
                 AND gsc_impressions > 0
            THEN gsc_clicks * 1.0 / gsc_impressions
            ELSE NULL
        END AS ctr

    FROM latest_feb
    WHERE rn = 1
),

scored AS (
    SELECT
        *,

        CASE
            WHEN gsc_data_available = TRUE
                 AND impressions >= 500
                 AND ctr < 0.02
                 AND gsc_avg_position > 10
                THEN 2

            WHEN gsc_data_available = TRUE
                 AND impressions >= 500
                 AND (
                     ctr < 0.02
                     OR gsc_avg_position > 10
                 )
                THEN 1

            ELSE 0
        END AS score,

        CASE
            WHEN gsc_data_available = TRUE
                 AND impressions >= 500
                 AND ctr < 0.02
                 AND gsc_avg_position > 10
                THEN 'visible_low_ctr_poor_position'

            WHEN gsc_data_available = TRUE
                 AND impressions >= 500
                 AND ctr < 0.02
                THEN 'visible_low_ctr'

            WHEN gsc_data_available = TRUE
                 AND impressions >= 500
                 AND gsc_avg_position > 10
                THEN 'visible_poor_position'

            ELSE 'insufficient_signal'
        END AS reason_code

    FROM base
)

SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    impressions,
    clicks,
    ROUND(ctr, 4) AS ctr,
    ROUND(gsc_avg_position, 2) AS avg_position,
    score,
    reason_code,

    CASE
        WHEN reason_code = 'visible_low_ctr_poor_position'
            THEN 'review_ctr_and_position'
        WHEN reason_code = 'visible_low_ctr'
            THEN 'review_low_ctr'
        WHEN reason_code = 'visible_poor_position'
            THEN 'review_position'
        ELSE 'monitor'
    END AS action

FROM scored

ORDER BY score DESC, impressions DESC, content_hash_id
"""

output_path = "/content/work/outputs/baseline_action_score.csv"

con.execute(f"""
COPY ({queue_sql})
TO '{output_path}'
WITH (HEADER, DELIMITER ',')
""")

print("Ranked queue written to:")
print(output_path)

# Show the first 20 ranked rows for the next review section.
top20 = con.execute(f"""
SELECT *
FROM ({queue_sql})
LIMIT 20
""").fetchdf()

display(top20)# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Ranked queue written to:
/content/work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Review the top 20 ranked recommendations.

top20_review = con.execute(f"""
SELECT *
FROM ({queue_sql})
LIMIT 20
""").fetchdf()

def confidence_note(row):
    if row["score"] == 2:
        return "high — both signals are present"
    elif row["score"] == 1:
        return "medium — one signal is present"
    else:
        return "low — insufficient signal"

def wrong_if(row):
    if row["reason_code"] == "visible_low_ctr_poor_position":
        return "could be wrong if low CTR is expected for the query intent or position data is unreliable"
    elif row["reason_code"] == "visible_low_ctr":
        return "could be wrong if low CTR is expected for the query intent"
    elif row["reason_code"] == "visible_poor_position":
        return "could be wrong if the page is intentionally targeting non-page-1 queries"
    else:
        return "could be wrong because the available search signal is incomplete"

top20_review["confidence_note"] = top20_review.apply(
    confidence_note, axis=1
)

top20_review["what_would_make_it_wrong"] = top20_review.apply(
    wrong_if, axis=1
)

review = top20_review[
    [
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

print("Top-20 review completed.")

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

The baseline is intentionally simple, so some top-ranked pages may be weak picks. A recommendation could be wrong when low CTR is expected for the query intent, when average position is unreliable, or when the page has incomplete search data.

Leakage check:

The score uses only February 2026 search signals: impressions, clicks, CTR, and average position. It does not use March 2026 data, product flags, or future-window labels.

In [ ]:
# Weak-pick review and leakage check

# Show a few lower-priority recommendations that may be weaker.
weak_picks = con.execute(f"""
SELECT
    content_hash_id,
    impressions,
    ROUND(ctr, 4) AS ctr,
    ROUND(avg_position, 2) AS avg_position,
    score,
    reason_code,
    action
FROM ({queue_sql})
WHERE score > 0
ORDER BY score ASC, impressions ASC
LIMIT 5
""").fetchdf()

print("Examples of weaker positive picks:")
display(weak_picks)

# Leakage checks on the rule definition.
sql_text = queue_sql.lower()

print("\nLeakage check:")

print("March/future window used:",
      "march" in sql_text)

print("Product flags used:",
      "product" in sql_text)

print("Future labels used:",
      "label" in sql_text)

print("\nFeature window: February 2026 only")
print("Expected result for all three checks: False")# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.